In [6]:
import os
import math
import warnings
import numpy as np
from pymatgen.core import Structure, Lattice
from pymatgen.analysis.interfaces.zsl import ZSLGenerator
from pymatgen.analysis.interfaces.coherent_interfaces import CoherentInterfaceBuilder
from pymatgen.io.lammps.data import LammpsData
from pymatgen.core.surface import SlabGenerator

warnings.filterwarnings("ignore")

# ================= 🔧 参数配置区 =================
structures_dir = "Na_Na3SbS4_Data/structures"
output_dir = "Na_Na3SbS4_Data/Interface_MD_ManualZ"

# 1. Z轴高度控制 (Å)
TARGET_THICKNESS_NA = 15.0     
TARGET_THICKNESS_ELYTE = 15.0  

# 2. XY轴控制
MIN_ATOMS_XY = 100    # 最小原子数限制
MIN_XY_LENGTH = 12.0  # [新增要求] 最小边长限制 (建议 > 2倍截断半径)

MAX_ATOMS = 200       # 总原子数上限 (防止爆内存)

SCAN_MILLERS = [(0, 0, 1)]
# =================================================

def find_structure_by_formula(folder, part_name):
    if not os.path.exists(folder): return None
    for fname in os.listdir(folder):
        if part_name in fname and fname.endswith(".vasp"):
            return os.path.join(folder, fname)
    return None

def count_layers_z(structure, tol=0.8):
    if not structure or len(structure) == 0: return 0
    z_coords = sorted([site.coords[2] for site in structure.sites])
    unique = [z_coords[0]]
    for z in z_coords[1:]:
        if abs(z - unique[-1]) > tol: unique.append(z)
    return len(unique)

def create_interface_final():
    print(f"🚀 开始生成界面...")
    print(f"   目标 Z 高度: Na >= {TARGET_THICKNESS_NA} Å, Elyte >= {TARGET_THICKNESS_ELYTE} Å")
    print(f"   目标 XY 尺寸: >= {MIN_XY_LENGTH} Å")

    f_na = find_structure_by_formula(structures_dir, "Na_mp-127")
    f_elyte = find_structure_by_formula(structures_dir, "Na3SbS4")
    
    if not f_na or not f_elyte:
        print("❌ 错误：找不到输入文件。")
        return None

    s1 = Structure.from_file(f_na).to_conventional()
    s2 = Structure.from_file(f_elyte).to_conventional()
    
    zsl = ZSLGenerator(
        max_area_ratio_tol=0.2, max_area=200, 
        max_length_tol=0.1, max_angle_tol=0.1 
    )
    
    best_struct = None
    best_final_atoms = float('inf') 
    best_info = ""
    final_stats = {}

    for hkl_sub in SCAN_MILLERS:
        for hkl_film in SCAN_MILLERS:
            try:
                # --- A. Z轴手动控制 (切薄片 -> 算倍数 -> 强制Z扩胞) ---
                base_na = SlabGenerator(s1, hkl_sub, min_slab_size=0.1, min_vacuum_size=0, center_slab=True).get_slab()
                base_elyte = SlabGenerator(s2, hkl_film, min_slab_size=0.1, min_vacuum_size=0, center_slab=True).get_slab()
                
                n_z_na = max(2, math.ceil(TARGET_THICKNESS_NA / base_na.lattice.c))
                n_z_elyte = max(2, math.ceil(TARGET_THICKNESS_ELYTE / base_elyte.lattice.c))
                
                thick_na = base_na.copy()
                thick_na.make_supercell([1, 1, n_z_na])
                thick_elyte = base_elyte.copy()
                thick_elyte.make_supercell([1, 1, n_z_elyte])
                
                # --- B. 构建界面 ---
                builder = CoherentInterfaceBuilder(
                    substrate_structure=thick_na, 
                    film_structure=thick_elyte,
                    substrate_miller=hkl_sub, 
                    film_miller=hkl_film,
                    zslgen=zsl,
                )
            except Exception:
                continue

            for term in builder.terminations:
                try:
                    for iface in builder.get_interfaces(termination=term):
                        
                        # =================================================
                        # 🟢 [关键修改] 结合 原子数要求 和 边长要求
                        # =================================================
                        n_prim = len(iface)
                        lat = iface.lattice
                        
                        # 1. 计算满足"最小原子数"需要的扩胞倍数
                        if n_prim >= MIN_ATOMS_XY:
                            s_atoms = 1
                        else:
                            s_atoms = math.ceil(math.sqrt(MIN_ATOMS_XY / n_prim))
                        
                        # 2. 计算满足"最小边长"需要的扩胞倍数
                        # 我们希望 a 和 b 都 > MIN_XY_LENGTH
                        s_len_a = math.ceil(MIN_XY_LENGTH / lat.a)
                        s_len_b = math.ceil(MIN_XY_LENGTH / lat.b)
                        s_length = max(s_len_a, s_len_b)
                        
                        # 3. 取两者中的最大值，确保两个条件都满足
                        s = max(s_atoms, s_length, 1)
                        # =================================================
                        
                        n_final = n_prim * (s ** 2)
                        
                        if n_final > MAX_ATOMS: continue 
                        
                        if n_final < best_final_atoms:
                            best_final_atoms = n_final
                            
                            temp_struct = iface.copy()
                            if s > 1:
                                temp_struct.make_supercell([s, s, 1])
                                
                            best_struct = temp_struct
                            best_info = f"Na{hkl_sub}//Elyte{hkl_film}"
                            
                            final_stats = {
                                "na_repeats": n_z_na, "elyte_repeats": n_z_elyte,
                                "s_xy": s
                            }
                            print(f"⭐ 候选: {best_info} | 扩胞 {s}x{s} | 边长: {lat.a*s:.1f}x{lat.b*s:.1f} Å | 原子: {n_final}")
                            
                except Exception:
                    continue

    print("-" * 60)
    
    if best_struct:
        os.makedirs(output_dir, exist_ok=True)
        print(f"🏆 最终选择: {best_info}")

        # ========================================================
        # 🟢 [后处理] 去真空 + 归一化 + 居中
        # ========================================================
        print("🔧 后处理: 去除真空 & 强制归一化...")
        
        # 1. 去真空 (Z-Compress)
        new_coords = best_struct.cart_coords.copy()
        new_coords[:, 2] -= np.min(new_coords[:, 2]) # 沉底
        
        physical_height = np.max(new_coords[:, 2])
        INTERFACE_GAP = 1.0
        new_c = physical_height + INTERFACE_GAP
        
        old_lat = best_struct.lattice
        new_lattice = Lattice.from_parameters(old_lat.a, old_lat.b, new_c, old_lat.alpha, old_lat.beta, old_lat.gamma)
        
        best_struct = Structure(new_lattice, best_struct.species, new_coords, coords_are_cartesian=True)
        
        # 2. 强制 Wrap (解决 Ovito 盒子外问题)
        frac_coords = best_struct.frac_coords % 1.0
        best_struct = Structure(new_lattice, best_struct.species, frac_coords, coords_are_cartesian=False)
        
        # 3. 居中
        best_struct.translate_sites(range(len(best_struct)), [0, 0, 0.5], to_unit_cell=True)

        # 保存
        v_path = os.path.join(output_dir, "Final_Structure.vasp")
        best_struct.to(filename=v_path, fmt="poscar")
        
        l_path = os.path.join(output_dir, "Final_Structure.data")
        ld = LammpsData.from_structure(best_struct, atom_style="atomic")
        ld.write_file(l_path)
        
        print(f"✅ 文件已保存: {v_path}")
        
        # 统计输出
        lat = best_struct.lattice
        comp = best_struct.composition
        
        print("-" * 30)
        print("📊 最终结果统计:")
        print(f"📏 尺寸: {lat.a:.2f} x {lat.b:.2f} x {lat.c:.2f} Å")
        print(f"🧮 原子: Na={int(comp['Na'])}, Sb={int(comp['Sb'])}, S={int(comp['S'])}")
        
        # 验证边长是否达标
        if lat.a < MIN_XY_LENGTH or lat.b < MIN_XY_LENGTH:
            print(f"⚠️ 警告: 边长小于 {MIN_XY_LENGTH} Å！")
        else:
            print(f"✅ XY边长达标 (>= {MIN_XY_LENGTH} Å)")
            
        print("-" * 30)

    else:
        print("❌ 未找到合适结构。")

if __name__ == "__main__":
    create_interface_final()

🚀 开始生成界面...
   目标 Z 高度: Na >= 15.0 Å, Elyte >= 15.0 Å
   目标 XY 尺寸: >= 12.0 Å
⭐ 候选: Na(0, 0, 1)//Elyte(0, 0, 1) | 扩胞 1x1 | 边长: 12.6x12.6 Å | 原子: 156
------------------------------------------------------------
🏆 最终选择: Na(0, 0, 1)//Elyte(0, 0, 1)
🔧 后处理: 去除真空 & 强制归一化...
✅ 文件已保存: Na_Na3SbS4_Data/Interface_MD_ManualZ/Final_Structure.vasp
------------------------------
📊 最终结果统计:
📏 尺寸: 12.62 x 12.62 x 28.19 Å
🧮 原子: Na=96, Sb=12, S=48
✅ XY边长达标 (>= 12.0 Å)
------------------------------


In [7]:
import os
import math
import warnings
from pymatgen.core import Structure
from pymatgen.io.vasp.sets import MPRelaxSet
from pymatgen.io.vasp.inputs import Kpoints

# ================= 配置区域 =================

input_dir = "Na_Na3SbS4_Data/Interface_MD_ManualZ"
base_output_dir = "Na_Na3SbS4_Data/Interface_AIMD_runs"
os.makedirs(base_output_dir, exist_ok=True)

# AIMD 参数设置 (保持你的设置不变)
aimd_settings = {
    "IBRION": 0,          
    "NSW": 5000,          
    "POTIM": 2.0,         
    "TEBEG": 800,         
    "TEEND": 800,
    "ISYM": 0,            
    "SMASS": 0,           
    "ISIF": 2,            
    "KBLOCK": 1,          
    "ALGO": "Fast",       
    "PREC": "Normal",     
    "LREAL": "Auto",      
    "NELM": 100,          
    "ISMEAR": -1,         # Fermi Smearing (适合金属界面)
    "SIGMA": 0.1,         
    "ISPIN": 1,           
    "LWAVE": False,       
    "LCHARG": False,      
    "NCORE": 8,           
}

# ================= 处理流程 =================
print(f"📂 读取结构目录: {input_dir} ...")

if not os.path.exists(input_dir):
    print(f"❌ 错误: 找不到目录 {input_dir}")
else:
    files = [f for f in os.listdir(input_dir) if f.endswith(('.cif', '.vasp'))]
    files.sort() 
    print(f"🔍 发现 {len(files)} 个结构文件。")

    for filename in files:
        file_path = os.path.join(input_dir, filename)
        struct_name = os.path.splitext(filename)[0]
        
        try:
            # 1. 加载结构
            structure = Structure.from_file(file_path)
            
            # 2. 扩胞检查
            min_length = 10.0
            lengths = structure.lattice.abc
            scaling_matrix = [max(1, int(math.ceil(min_length / l))) for l in lengths]
            
            if any(x > 1 for x in scaling_matrix):
                print(f"  - {struct_name}: 执行扩胞 {scaling_matrix}")
                structure.make_supercell(scaling_matrix)
            
            # 3. 创建目录
            task_dir = os.path.join(base_output_dir, struct_name)
            os.makedirs(task_dir, exist_ok=True)
            
            # 4. 生成 VASP 输入文件 (修复点在这里!)
            # 我们先创建 Kpoints 对象
            gamma_only = Kpoints.gamma_automatic()
            
            # 然后直接传给 MPRelaxSet
            vis = MPRelaxSet(
                structure, 
                user_incar_settings=aimd_settings,
                user_potcar_functional="PBE",
                user_kpoints_settings=gamma_only  # <--- 直接在这里传入对象
            )
            
            # 写入文件
            vis.write_input(task_dir)
            print(f"✅ 生成成功: {struct_name} (原子数: {structure.num_sites}) -> {task_dir}")
            
        except Exception as e:
            print(f"❌ 跳过 {filename}: {e}")

    print(f"\n🎉 全部完成！请检查 '{base_output_dir}'")

📂 读取结构目录: Na_Na3SbS4_Data/Interface_MD_ManualZ ...
🔍 发现 1 个结构文件。
✅ 生成成功: Final_Structure (原子数: 156) -> Na_Na3SbS4_Data/Interface_AIMD_runs/Final_Structure

🎉 全部完成！请检查 'Na_Na3SbS4_Data/Interface_AIMD_runs'


In [ ]:
import os
from dpdispatcher import Machine, Resources, Task, Submission

# ================= 配置区域 =================

# 1. 基础路径
work_base = "Na_Na3SbS4_Data/Interface_AIMD_runs" 

# 2. 指定你要跑的那个文件夹名字 (精准打击)
target_folder_name = "Final_Structure"

# 3. VASP 路径
vasp_exe = "/dssg/opt/icelake/linux-centos8-icelake/oneapi-2021.4.0/vasp/vasp.6.3.0/bin/vasp_std"

# ================= 机器与资源 =================

machine = Machine(
    batch_type="Slurm",
    context_type="LazyLocal",
    local_root="./",
)

resources = Resources(
    number_node=1,
    cpu_per_node=64,
    gpu_per_node=0,
    queue_name="64c512g", 
    group_size=1,
    module_list=["vasp/6.3.0-intel-2021.4.0"], 
    custom_flags=[
        "#SBATCH --partition=64c512g",
        "#SBATCH --ntasks=64",
        "#SBATCH --mail-type=end",
        "#SBATCH --mail-user=2393474010@sjtu.edu.cn",
        "#SBATCH --time=120:00:00",      
        f"#SBATCH --job-name={target_folder_name}" # 任务名直接用文件夹名
    ]
)

setup_env = (
    "ulimit -s unlimited && "
    "ulimit -l unlimited && "
    "export I_MPI_PMI_LIBRARY=/usr/lib64/libpmi.so && "
    "export I_MPI_FABRICS=shm:ofi && "
    "export I_MPI_PMI=pmi"
)
command = f"{setup_env} && mpirun -n 64 {vasp_exe}"

# ================= 构建任务 =================

task_list = []
full_path = os.path.join(work_base, target_folder_name)

print(f"🎯 正在定位目标任务: {full_path} ...")

# 检查文件夹是否存在，并且里面有 INCAR
if os.path.isdir(full_path) and os.path.exists(os.path.join(full_path, "INCAR")):
    print(f"  ✅ 找到任务文件夹，准备提交...")
    
    task = Task(
        command=command,
        task_work_path=target_folder_name, # 注意：这里填相对路径(文件夹名)即可，因为 Submission 指定了 work_base
        forward_files=['INCAR', 'POSCAR', 'POTCAR', 'KPOINTS'], 
        backward_files=['OUTCAR', 'vasprun.xml', 'OSZICAR', 'XDATCAR', 'CONTCAR'] 
    )
    task_list.append(task)
else:
    print(f"❌ 错误: 找不到文件夹 {full_path} 或其中缺少 INCAR 文件！")

# ================= 提交执行 =================

if len(task_list) > 0:
    submission = Submission(
        work_base=work_base,
        machine=machine,
        resources=resources,
        task_list=task_list
    )
    
    submission.run_submission()
    print(f"🚀 任务 {target_folder_name} 已提交！")
    print("📋 请使用 'squeue' 查看状态。")
else:
    print("❌ 提交终止。")

In [1]:
import os
import numpy as np
from pymatgen.core import Structure
from pymatgen.analysis.interfaces.zsl import ZSLGenerator
from pymatgen.analysis.interfaces.coherent_interfaces import CoherentInterfaceBuilder

# ================= 配置区域 =================
structures_dir = "Na_Na3SbS4_Data/structures" # 请确保路径正确
# 定义要扫描的晶面 (低指数晶面通常最稳定)
SCAN_MILLERS = [
    (0, 0, 1), 
    (1, 0, 0),
    (1, 1, 0),
    (1, 1, 1)
]

# ================= 辅助函数 =================
def find_file(folder, substring):
    for f in os.listdir(folder):
        if substring in f and f.endswith(".vasp"):
            return os.path.join(folder, f)
    return None

def test_interface_finding():
    print("🚀 开始界面匹配测试 (只读模式)...")
    
    # 1. 读取文件
    f_na = find_file(structures_dir, "Na_mp-127")
    f_elyte = find_file(structures_dir, "Na3SbS4")
    
    if not f_na or not f_elyte:
        print("❌ 错误：找不到结构文件，请检查路径。")
        return

    print(f"📖 读取结构:\n  - {os.path.basename(f_na)}\n  - {os.path.basename(f_elyte)}")
    s1 = Structure.from_file(f_na).to_conventional()
    s2 = Structure.from_file(f_elyte).to_conventional()

    # 2. 设置 ZSL (这是能否找到结构的关键)
    # 只要满足这些几何条件，Builder 就会生成结构
    zsl = ZSLGenerator(
        max_area_ratio_tol=0.2, 
        max_area=600,         # 如果找不到，尝试增加这个值到 800 或 1000
        max_length_tol=0.15,  # 15% 边长容差
        max_angle_tol=0.15    # 15% 角度容差
    )

    found_any = False
    
    print("-" * 60)
    print(f"{'基底(Na)':<15} | {'薄膜(Elyte)':<15} | {'终端':<15} | {'结果'}")
    print("-" * 60)

    # 3. 循环扫描
    for hkl_sub in SCAN_MILLERS:
        for hkl_film in SCAN_MILLERS:
            
            try:
                builder = CoherentInterfaceBuilder(
                    substrate_structure=s1,
                    film_structure=s2,
                    substrate_miller=hkl_sub,
                    film_miller=hkl_film,
                    zslgen=zsl
                )
            except Exception:
                # 晶面无法切出（例如几何上不允许），跳过
                continue

            # 遍历终端
            for term in builder.terminations:
                try:
                    # 尝试获取界面
                    interfaces = list(builder.get_interfaces(termination=term))
                    
                    if len(interfaces) > 0:
                        # 只要列表不为空，就说明找到了！
                        found_any = True
                        
                        # 哪怕不知道应变具体是多少，ZSL保证了它一定小于 15%
                        # 我们打印出原子数作为参考
                        num_atoms = len(interfaces[0]) 
                        print(f"{str(hkl_sub):<15} | {str(hkl_film):<15} | {str(term):<15} | ✅ 成功 (原子数: {num_atoms})")
                        
                except Exception:
                    continue

    print("-" * 60)
    if found_any:
        print("🎉 测试通过！当前 ZSL 参数可以找到匹配的界面结构。")
        print("建议：选择上面原子数较少（例如 < 200）的组合进行生成。")
    else:
        print("❌ 测试失败。在所有扫描的晶面中都未找到匹配。")
        print("建议修改 ZSLGenerator 参数：")
        print("  1. 增大 max_area (例如 600 -> 1000)")
        print("  2. 增大 max_length_tol (例如 0.15 -> 0.20)")

if __name__ == "__main__":
    test_interface_finding()

🚀 开始界面匹配测试 (只读模式)...
📖 读取结构:
  - Na_mp-127.vasp
  - Na3SbS4_mp-10167.vasp
------------------------------------------------------------
基底(Na)          | 薄膜(Elyte)       | 终端              | 结果
------------------------------------------------------------
(0, 0, 1)       | (0, 0, 1)       | ('Na3Sb_P4/mmm_4', 'Na_P4/mmm_1') | ✅ 成功 (原子数: 42)
(0, 0, 1)       | (0, 0, 1)       | ('S_Cmmm_2', 'Na_P4/mmm_1') | ✅ 成功 (原子数: 42)
(0, 0, 1)       | (1, 0, 0)       | ('Na3Sb_P4/mmm_4', 'Na_P4/mmm_1') | ✅ 成功 (原子数: 42)
(0, 0, 1)       | (1, 0, 0)       | ('S_Cmmm_2', 'Na_P4/mmm_1') | ✅ 成功 (原子数: 42)
(0, 0, 1)       | (1, 1, 0)       | ('NaSbS2_Amm2_4', 'Na_P4/mmm_1') | ✅ 成功 (原子数: 24)
(0, 0, 1)       | (1, 1, 0)       | ('S_Cmmm_1', 'Na_P4/mmm_1') | ✅ 成功 (原子数: 24)
(0, 0, 1)       | (1, 1, 1)       | ('Na3Sb_R-3m_4', 'Na_P4/mmm_1') | ✅ 成功 (原子数: 48)
(0, 0, 1)       | (1, 1, 1)       | ('S_R3m_3', 'Na_P4/mmm_1') | ✅ 成功 (原子数: 48)
(0, 0, 1)       | (1, 1, 1)       | ('S_R-3m_1', 'Na_P4/mmm_1') | ✅ 成功 (原子数: 48

In [24]:
import os
import math
import warnings
import numpy as np
from pymatgen.core import Structure
from pymatgen.analysis.interfaces.zsl import ZSLGenerator
from pymatgen.analysis.interfaces.coherent_interfaces import CoherentInterfaceBuilder
from pymatgen.io.lammps.data import LammpsData

warnings.filterwarnings("ignore")

# --- CONFIGURATION ---
structures_dir = "Na_Na3SbS4_Data/structures"
output_dir = "Na_Na3SbS4_Data/Interface_MD_Manual"
min_atoms = 100  # Target size

# UPDATED SCAN LIST
# Includes (1,1,1) for FCC-like LiGa and various low-index planes
# SCAN_MILLERS = [
#     (1, 1, 1),  # Densest plane for LiGa (Cubic)
#     (1, 1, 0),
#     (1, 0, 0),
#     (0, 0, 1),
#     (2, 1, 0),  # Search slightly higher indices for Li2Ga match
#     (2, 1, 1)
# ]

SCAN_MILLERS = [
    (0, 0, 1),
]

def find_structure_by_formula(folder, formula):
    """ Finds the first CIF file in the folder matching the formula. """
    for fname in os.listdir(folder):
        if fname.endswith(".vasp") and formula in fname:
            return os.path.join(folder, fname)
    return None

def create_interface():
    print("Searching for LiGa and Li2Ga structures...")
    file_LiGa = find_structure_by_formula(structures_dir, "Na")
    file_Li2Ga = find_structure_by_formula(structures_dir, "Na3SbS4")
    
    if not file_LiGa or not file_Li2Ga:
        print(f"Error: Could not find LiGa or Li2Ga files in {structures_dir}")
        return None

    print(f"Found: {os.path.basename(file_LiGa)} & {os.path.basename(file_Li2Ga)}")
    
    s1 = Structure.from_file(file_LiGa).to_conventional()
    s2 = Structure.from_file(file_Li2Ga).to_conventional()
    
    # Relaxed Tolerances for Strain Matching
    zsl = ZSLGenerator(
        max_area_ratio_tol=0.2, # Allow 25% area mismatch (flexible)
        max_area=400,
        max_length_tol=0.15,     # 15% length mismatch
        max_angle_tol=0.15       # 15% angle mismatch
    )
    
    best_interface = None
    min_strain = float('inf')
    best_orientation = None

    print(f"\nScanning Miller Index Space: {len(SCAN_MILLERS)} planes")
    print("-" * 60)

    # --- LOOP OVER ORIENTATIONS ---
    # We check every combination of substrate/film orientation
    for hkl_sub in SCAN_MILLERS:
        for hkl_film in SCAN_MILLERS:
            
            # Skip redundant checks if simple cubic symmetry implies they are same
            # (But safe to check for general structures)
            
            try:
                builder = CoherentInterfaceBuilder(
                    substrate_structure=s1,
                    film_structure=s2,
                    film_miller=hkl_film, 
                    substrate_miller=hkl_sub,
                    zslgen=zsl
                )
                
                # Get interfaces (no termination constraint)
                interfaces = list(builder.get_interfaces())
                
                if not interfaces:
                    continue
                
                # Find lowest strain in this batch
                interfaces.sort(key=lambda x: x.separation)
                current_best = interfaces[0]
                strain = current_best.separation
                
                print(f"Match: LiGa {hkl_sub} // Li2Ga {hkl_film} -> Strain: {strain:.4f} Å")
                
                if strain < min_strain:
                    min_strain = strain
                    best_interface = current_best
                    best_orientation = f"LiGa {hkl_sub} // Li2Ga {hkl_film}"
                    
            except Exception:
                continue

    print("-" * 60)
    
    if best_interface:
        print(f"BEST INTERFACE: {best_orientation}")
        print(f"Strain: {min_strain:.4f} Å")
        
        struct = best_interface.copy()
        
        # Scale to Target Size
        current_atoms = len(struct)
        if current_atoms < min_atoms:
            scale_factor = (min_atoms / current_atoms) ** (1/2)
            s = math.ceil(scale_factor)
            # Elongate Z to ensure bulk-like regions away from interface
            struct.make_supercell([s, s, 1]) 
        
        print(f"Final Supercell: {len(struct)} atoms")
        return struct
    else:
        print("Failed to find any coherent interface. Try increasing 'max_area' in ZSLGenerator.")
        return None

# --- MAIN EXECUTION ---
os.makedirs(output_dir, exist_ok=True)
interface_struct = create_interface()

if interface_struct:
    # Write Data
    ld = LammpsData.from_structure(interface_struct, atom_style="atomic")
    ld.write_file(os.path.join(output_dir, "data.lammps"))
    
    print(f"\nSuccess! Interface data written to {os.path.join(output_dir, 'data.lammps')}")
    
    elements = sorted(list(interface_struct.composition.get_el_amt_dict().keys()))
    print("Atom Type Mapping:")
    for i, el in enumerate(elements):
        print(f"  Type {i+1} = {el}")
else:
    print("Generation Failed.")

Searching for LiGa and Li2Ga structures...
Found: Na_mp-974558.vasp & Na3SbS4_mp-10167.vasp

Scanning Miller Index Space: 1 planes
------------------------------------------------------------
------------------------------------------------------------
Failed to find any coherent interface. Try increasing 'max_area' in ZSLGenerator.
Generation Failed.
